# DeiT seed2 — E0/E1/E2/E3/E4 expansion-generator diagnostics

Attach CIFAR-100 and the original full seed2 fork: epoch375 / validation56.20%.
Attach the previous D0 output archive/result if available; it must match fork hash,
protocol and CP config. Without it, D0 all12 is recomputed first and explicitly
labelled a fresh reference, not verification against a missing old artifact.
Enable Internet, a GPU, and `github_token`. This notebook does not train A3/A4,
run150 epochs, apply a correction, or update the original optimizer.

E0 reproduces all12 native rank4 TINY WHERE gains/norm/rank before proceeding.
Default selected sites: TINY top1, smallest D0 projection residual, median TINY
rank; deterministic fillers produce three distinct sites if selections overlap.
E3/E4 run first for all selected sites, followed by E1 ranks(2,4,8,16) and E2
probe epsilons(.01,.05,.1,.2) × expansion scales(.25,.5,1,2). Duplicate cases share
one proposal/solve. Set ALL_BLOCKS=True to deliberately extend the suite to12;
there is no automatic selection based on validation or a new acceptance threshold.

Generators: original TINY; temporary A/a/B optimized on disjoint opt_fit64 for20
Adam steps (A nonzero, B=0 initially); random auxiliary neurons normalized to the
TINY projection-logit norm. Failed/degenerate norm matching is labelled. No
comparison is claimed to match compute; the unique-proposal budget records
forward/backward counts, wall times and seeds.

`probe_epsilon` is the existing virtual expansion measurement gate; native TINY
proposal generation depends on rank/statistics, not this epsilon. Expansion scale
is a separate control. Default exact_expanded_forward reruns the entire expanded
network with gate=epsilon*scale; linear_logits separately implements scale*dE.
Gradient-tuned branches are trained/evaluated at gate1; this difference is logged.

FP32/FP64 CE metrics use the same original forward logits. Default additionally
recomputes the whole frozen model and fixed proposal in FP64; this does not
recover errors incurred while constructing the original FP32 TINY proposal.
Validation is never evaluated or used to choose a method/hyperparameter. WHERE,
projection, gate and tuning roles remain distinct; gate is a selection diagnostic.
Every projection affects only the original MLP of the expansion's own site and
is previewed without commit. Model/Parameter/optimizer/RNG invariance is checked.

Outputs: expansion_diagnostics.csv, existing_tiny_diagnostics.csv, configuration,
summary, unique-proposal budget, actual probe tensors/indices, cached proposals
and diagnostic_progress.pt. Resume skips committed diagnostic rows, not training
arms. Download the archive for another Kaggle session.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_deit_projection_aware.py", "tests/test_deit_expansion_diagnostics.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import os
OUTPUT = Path('/kaggle/working/deit_seed2_expansion_diagnostics_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))

ALL_BLOCKS = False
VERIFY_FORWARD_FP64 = True
AUXILIARY_STEPS = 20


In [ ]:
from experiments.run_deit_expansion_diagnostics import prepare_inputs
FORK, _, _, D0 = prepare_inputs(['/kaggle/input'], OUTPUT)
print('Fork:', FORK, 'Existing D0:', D0 is not None)


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_expansion_diagnostics',
    '--data-root', str(DATA_ROOT), '--output', str(OUTPUT), '--input-root', '/kaggle/input',
    '--auxiliary-steps', str(AUXILIARY_STEPS)]
if ALL_BLOCKS: command += ['--all-blocks']
if not VERIFY_FORWARD_FP64: command += ['--no-forward-fp64']
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable diagnostics:', archive)


In [ ]:
print((OUTPUT / 'summary.json').read_text())
import pandas as pd
frame = pd.read_csv(OUTPUT / 'expansion_diagnostics.csv')
display(frame[['proposal_method','block_name','rank','probe_epsilon','expansion_scale',
    'e_gain_fp32','e_gain_fp64','e_gain_forward_fp64','projection_residual','projection_cosine','gate_gain','accepted']])
